*Title:* 1_FST_Generation

*Goal:* Reorganize the file and get information on coverage. Then, generate pairwise fst (per SNP), and have a dataset of SNPs present in both years. Then, calculate outliers (5% FST), and determine repeatabililty.

Corresponds to poolfstat_onward.md script 

# 1.1 Reorder file
*Goal:* Split the sync file for each year and reorder the populations so that each estuary-timepoint is nearby. Then, calculate the coverage per year (as each sequenced separately)

# Split File:
Order based on bam_list (and thus sync file order, generated in 0_Assembly)
``` bash
awk 'BEGIN {OFS="\t"} {print $1, $2, $3, $27, $20, $21, $18, $23, $24, $17, $19, $26, $25, $16, $22}' masked-autosome-chrom.sync > 1.2016masked-autosome-chrom-reorder.sync
echo "2016 done"
awk 'BEGIN {OFS="\t"} {print $1, $2, $3, $14, $4, $7, $11, $10, $5, $12, $6, $13, $8, $15, $9}' masked-autosome-chrom.sync > 1.2017masked-autosome-chrom-reorder.sync
echo "2017 done"
awk 'BEGIN {OFS="\t"} {print $1, $2, $3, $14, $4, $7, $11, $10, $5, $12, $6, $13, $8, $15, $9, $27, $20, $21, $18, $23, $24, $17, $19, $26, $25, $16, $22}' masked/masked-autosome-chrom.sync > 1.2017_2016masked-autosome-chrom-reorder.sync
echo "all done"
```
#The order of 2016:
"Younger_Spring-2016","Younger_Fall-2016","OldDairy_Spring-2016","OldDairy_Fall-2016","Scott_Spring-2016","Scott_Fall-2016","Lombardi_Spring-2016",
"Lombardi_Fall-2016","Laguna_Spring-2016","Laguna_Fall-2016","Waddell_Spring-2016","Waddell_Fall-2016"

#The order of 2017:
"Younger_Spring-2017","Younger_Fall-2017","OldDairy_Spring-2017","OldDairy_Fall-2017","Scott_Spring-2017","Scott_Fall-2017","Lombardi_Spring-2017",
"Lombardi_Fall-2017","Laguna_Spring-2017","Laguna_Fall-2017","Waddell_Spring-2017","Waddell_Fall-2017"


# Coverage:
Showing for 2016 only, but also ran for 2017:

``` bash
#2016- split and rename the files: 
# Extract columns 4-15 and save to a temporary file
cut -f4-15 1.2016masked-autosome-chrom-reorder.sync > 2.a.2016histogram-data-preheader.sync
echo "preheader file complete"

# Define the header
echo -e "Younger_Spring-2016\tYounger_Fall-2016\tOldDairy_Spring-2016\tOldDairy_Fall-2016\tScott_Spring-2016\tScott_Fall-2016\tLombardi_Spring-2016\tLombardi_Fall-2016\tLaguna_Spring-2016\tLaguna_Fall-2016\tWaddell_Spring-2016\tWaddell_Fall-2016" > 2.a.2016header.txt
echo "header file complete"

# Combine the header and the extracted columns into the final output file
cat 2.a.2016header.txt 2.a.2016histogram-data-preheader.sync > 2.a.2016histogram-data.sync

# Print a message indicating completion
echo "columns extracted and header made"
``` 

Get Allele counts, exxcluding 0
- Here I want to break the x:x:x:x allele counts. These counts will then be used to make histograms of coverage.
``` bash
echo "start part 2b"
awk -v OFS='\t' '
NR == 1 { #for the header row
    for (i = 1; i <= NF; i++) { #check all columns
        rename[i] = $i
    }
    print "Population", "CoverageNumbers"
    }
NR > 1 { #for rows after the header
    for (i = 1; i <= NF; i++) { #check all columns
        n = split($i, nums, ":")
        for (j = 1; j <= n; j++) {
            if (nums[j] != 0) {
                print rename[i], nums[j]
            }
        }
    }
}
' 2.a.2016histogram-data.sync > 2.b.2016histogram-data.tsv

#Decrease the file size for histogram processing
awk 'BEGIN {OFS="\t"} {print $2}' 2.b.2016histogram-data.tsv  > 2.c.2016histogram-data.tsv

echo "2c file made"


#2016 version
#This is to find the min max (0, 3007):
file="2.c.2016histogram-data.tsv"; column=1; awk -v col=$column 'BEGIN { min = "unset"; max = "unset" } NR > 1 { value = $col; if (min == "unset" || value < min) { min = value } if (max == "unset" || value > max) { max = value } } END { print "Min:", min; print "Max:", max }' "$file" > 2.d.2016coverage.txt
min= 1 max = 3007


#Then this to get the histogram data 
#2016 data: (I ran it with 3 different bin size 10, 155, and 310, since i set the range to 0-3100):

file="2.c.2016histogram-data.tsv"; column=1; binwidth=10; min_value=0; max_value=3100; num_bins=$(( (max_value - min_value) / binwidth )); awk -v col=$column -v binwidth=$binwidth -v min_value=$min_value -v max_value=$max_value -v num_bins=$num_bins 'BEGIN { for (i = 0; i <= num_bins; i++) { hist_counts[i] = 0 } } { if (NR == 1) { next } value = $col; if (value >= min_value && value < max_value) { bin_index = int((value - min_value) / binwidth); hist_counts[bin_index]++ } } END { for (i = 0; i <= num_bins; i++) { print min_value + i * binwidth, hist_counts[i] } }' "$file" > 2.e.2016.bin10.histogram_data.txt

file="2.c.2016histogram-data.tsv"; column=1; binwidth=155; min_value=0; max_value=3100; num_bins=$(( (max_value - min_value) / binwidth )); awk -v col=$column -v binwidth=$binwidth -v min_value=$min_value -v max_value=$max_value -v num_bins=$num_bins 'BEGIN { for (i = 0; i <= num_bins; i++) { hist_counts[i] = 0 } } { if (NR == 1) { next } value = $col; if (value >= min_value && value < max_value) { bin_index = int((value - min_value) / binwidth); hist_counts[bin_index]++ } } END { for (i = 0; i <= num_bins; i++) { print min_value + i * binwidth, hist_counts[i] } }' "$file" > 2.e.2016.bin155.histogram_data.txt

file="2.c.2016histogram-data.tsv"; column=1; binwidth=310; min_value=0; max_value=3100; num_bins=$(( (max_value - min_value) / binwidth )); awk -v col=$column -v binwidth=$binwidth -v min_value=$min_value -v max_value=$max_value -v num_bins=$num_bins 'BEGIN { for (i = 0; i <= num_bins; i++) { hist_counts[i] = 0 } } { if (NR == 1) { next } value = $col; if (value >= min_value && value < max_value) { bin_index = int((value - min_value) / binwidth); hist_counts[bin_index]++ } } END { for (i = 0; i <= num_bins; i++) { print min_value + i * binwidth, hist_counts[i] } }' "$file" > 2.e.2016.bin310.histogram_data.txt
#note than Wing ran this himself (for 2016 only), so I just used his output files, rather than re-running it with mine.
``` 
Graph the data! (in R on local computer)
``` R
  # Load the necessary packages
  library(ggplot2)

# Read the histogram data. Running for different bin sizes:
hist_data <- read.table("2.e.2016.bin10.histogram_data.txt", header = FALSE, col.names = c("breaks", "counts"))
binwidth = 10 #adjust this to match the bin size of the file
#didn;t bother graphing the bigger bin files, because they don't give as much detailed info

# Plot the accumulated histogram
ggplot(hist_data, aes(x = breaks, y = counts)) +
  geom_bar(stat = "identity", width = binwidth, just =0) +
  theme(axis.text.x = element_text(size = 4, angle = 90, hjust = 0))+
  scale_x_continuous(breaks = seq(min(hist_data$breaks), max(hist_data$breaks), by = 50))
```
This gives us info for the coverage limit for FST.

# 1.2_Calculate FST 
*Goal:* Calculate FST per SNP. Showing for 2016 only, but 2017 was only run, except with a  "max.cov.per.pool = 300" and pool sizes of (58,58,60,56,58,76,60,60,60,56,42,60) for ("Younger_Spring-2017","Younger_Fall-2017","OldDairy_Spring-2017","OldDairy_Fall-2017","Scott_Spring-2017","Scott_Fall-2017","Lombardi_Spring-2017", "Lombardi_Fall-2017","Laguna_Spring-2017","Laguna_Fall-2017","Waddell_Spring-2017","Waddell_Fall-2017")

``` bash
module load r/4.4.0
Rscript 3.a.ii2016-pollfstat-fst.R
```

3.a.ii2016-pollfstat-fst.R:
``` R
module load r/4.4.0
library(poolfstat)
poolitem2016<-popsync2pooldata(
  sync.file = "1.2016masked-autosome-chrom-reorder.sync",
  poolsizes = c(80,80,80,80,80,80,80,80,80,80,80,80),
  poolnames = c("Younger_Spring-2016","Younger_Fall-2016","OldDairy_Spring-2016","OldDairy_Fall-2016","Scott_Spring-2016","Scott_Fall-2016","Lombardi_Spring-2016",
  "Lombardi_Fall-2016","Laguna_Spring-2016","Laguna_Fall-2016","Waddell_Spring-2016","Waddell_Fall-2016"),
  min.rc = 2,
  min.cov.per.pool = 5,
  max.cov.per.pool = 100,
  min.maf =-1,
  noindel = TRUE,
  nthreads = 10
)
save(poolitem2016, file = "3.a.ii2016-pollfstat-item.RData")
print("poolobject made")
PairwiseFST2016<-compute.pairwiseFST(
  poolitem2016,
  method = "Anova",
  min.cov.per.pool = 5,
  max.cov.per.pool = 100,
  min.maf = -1,
  output.snp.values = TRUE,
  nsnp.per.bjack.block = 0,
  verbose = TRUE
)

print("analysis done")
save(PairwiseFST2016, file = "3.a.ii2016-pollfstat-fst.RData")
print("done")
#output results for 2016: 6682020 SNPs for 12 Pools
#output results for 2017:10511906 SNPs for 12 Pools
```
# Preparing the FST files -------------------------------------------
*Goal:* HERE I want to get the SNP location data so I can put it together with the FST results. Ran for 2017 too but only showing 2016
```R
#For 2016:
library("poolfstat")
load("3.a.ii2016-pollfstat-item.RData")
SNPs<-poolitem2016@snp.info
write.table(SNPs, file = "3.a.ii2016-pollfstat-SNP-sites.tsv", sep = "\t", row.names = FALSE, quote = FALSE)
#Non-interactive:
module load r/4.4.0
Rscript 3.a.ii2016-pollfstat-tidy.R
library("poolfstat")
load("3.a.ii2016-pollfstat-fst.RData")
SNP_FST<-PairwiseFST2016@PairwiseSnpFST
print("FST data loaded")
write.table(SNP_FST, file = "3.a.ii2016-poolfstat-SNP-FST.tsv", sep = "\t", row.names = FALSE, quote = FALSE)
print("saved")
```
*Goal:* Combine SNP info
``` bash
module load r/4.4.0
Rscript 3.b.ii2016-poolfstat-tidy.R
```
RScript:
``` R
location <- read.csv("3.a.ii2016-pollfstat-SNP-sites.tsv", header= TRUE, sep = "\t")
FST<- read.csv("3.a.ii2016-poolfstat-SNP-FST.tsv", header= TRUE, sep = "\t")
print(colnames(FST)) #Double checking column names, because it looks like there might be some weird renaming via file conversion etc
Chromosome_Position <- paste(location$Chromosome, location$Position, sep = "_")  #Merging the location
FST_filtered <- data.frame(Chromosome_Position = Chromosome_Position) #making a dataframe for the columns I want 
#Instead of bringing all FST comparisons, I only want the relevant ones  
FST_filtered$Younger2016 <- FST$`Younger_Spring.2016.Younger_Fall.2016`
FST_filtered$OldDairy2016 <- FST$`OldDairy_Spring.2016.OldDairy_Fall.2016`
FST_filtered$Scott2016 <- FST$`Scott_Spring.2016.Scott_Fall.2016`
FST_filtered$Lombardi2016<- FST$`Lombardi_Spring.2016.Lombardi_Fall.2016`
FST_filtered$Laguna2016 <- FST$`Laguna_Spring.2016.Laguna_Fall.2016`
FST_filtered$Waddell2016 <- FST$`Waddell_Spring.2016.Waddell_Fall.2016`
write.table(FST_filtered, file = "3.b.ii2016-poolfstat-SNP-FST-with-locations.tsv", sep = "\t", row.names = FALSE, quote = FALSE, col.names= TRUE)
print("saved")
```
# Fst filter --------------------------------------------------------------
*Goal:* Only have sites that are present in both 2016 and 2017, and deal with populations with no variation (i.e., no alternate allele results in  NA which is FST=0, and negative FST values(which should be 0))
``` bash
awk 'FILENAME=="3.b.ii2017_poolfstat_SNP_FST_with_locations.tsv" {a[$1]; next} $1 in a' 3.b.ii2017_poolfstat_SNP_FST_with_locations.tsv 3.b.ii2016-poolfstat-SNP-FST-with-locations.tsv > 3.c.ii2016_poolfstat_SNP_FST_with_locations.tsv #getting 2016 fst results that match 2017
awk 'FILENAME=="3.c.ii2016_poolfstat_SNP_FST_with_locations.tsv" {a[$1]; next} $1 in a' 3.c.ii2016_poolfstat_SNP_FST_with_locations.tsv 3.b.ii2017_poolfstat_SNP_FST_with_locations.tsv > 3.c.ii2017_poolfstat_SNP_FST_with_locations.tsv #getting 2017 fst results that match 2016

#Quality Check:
wc -l 3.c.ii2016_poolfstat_SNP_FST_with_locations.tsv # results : 6261839
wc -l 3.c.ii2017_poolfstat_SNP_FST_with_locations.tsv #results : 6261839
```
Deal with 0:
``` R
library(data.table)
library(tidyverse)

# 2016 version:
file<- "3.c.ii2016_poolfstat_SNP_FST_with_locations.tsv"
data <- read.table(file, header = TRUE, sep = "\t")

#For FST columns, change NA to 0 and negatives to 0
data_mod <- data %>%
 mutate(across(everything(), ~ ifelse(is.na(.), 0, .)))

#Make negatives 0:
 data_mod <- data_mod %>%
  mutate(across(everything(), ~ ifelse(. < 0, 0, .)))

write.table(data_mod, "3.d.ii2016_poolfstat_SNP_FST_with_locations.tsv", sep = "\t", row.names = FALSE, quote = FALSE)

      #Quality Checking:
      head(data)
      head(data_mod) #looks correct
      min_values <- lapply(data, min, na.rm = TRUE)
      min_values <- lapply(data_mod, min, na.rm = TRUE) #no more negatives
      any_na <- anyNA(data)
      any_na <- anyNA(data_mod) #no more NA
      max_values <- lapply(data, max, na.rm = TRUE)
      max_values
      max_values <- lapply(data_mod, max, na.rm = TRUE) #max of 1, looks good
      max_values

      # Check rows that differ:
      data <- data %>%
        mutate(row_id = row_number())

      data_mod <- data_mod %>%
        mutate(row_id = row_number())

      differences <- data %>%
        filter(!row_id %in% data_mod$row_id[apply(data == data_mod, 1, all)]) #nrow= 6252994

        head(differences)
        diff<-differences %>% 
        drop_na() %>%  #nrow= 2559117 
        filter(if_all(everything(), ~ . >= 0)) #nrow = 0 (should be 0 and is 0). Means that any differences other than NA or negative don't exist
```
# FST outlier -------------------------------------------------------------
*Goal:* Calculate the FST outliers for each year and each population separately.
``` bash
module load r/4.4.0 
Rscript 4.OutlierDetection.R
``` 
RScript:
``` R
library(data.table)
library(tidyverse)
setwd("poolfstat-fst-version-ii") 

# List of files to loop over
files <- c("3.d.ii2016_poolfstat_SNP_FST_with_locations.tsv", "3.d.ii2017_poolfstat_SNP_FST_with_locations.tsv")

# Loop over the files
for (file in files) {
  # Read data
  FST_all <- fread(file, header = TRUE, data.table = FALSE)

long_data <- FST_all %>%
  pivot_longer(cols = -1, # All columns except the first (Chromosome_Position)
               names_to = "Population", 
               values_to = "Fst_value") #Turning the populations from columns to rows

# Step 3: Use pivot_wider to make Chromosome_Position the column names
wide_data <- long_data %>% #Making the cheom-position the column
  pivot_wider(names_from = Chromosome_Position, values_from = Fst_value) %>%
  column_to_rownames("Population")  # Set Population as row names 

wide_data <-as.data.frame(wide_data)
cutoff<- apply(wide_data, 1, quantile, probs = 0.95, na.rm = T)  
print(cutoff)
outliers_95 <- wide_data > cutoff 

output_filename <- paste0("4.outliers_", gsub(".tsv", "", file), "_95.csv")

write.csv(outliers_95, output_filename)
}
```

# 1.3 Generating datasets
*Goal:* Make a combined, separate, and outlier-only datasets

 Combined Dataset-------------------------------------------------------------
 
``` R
library(data.table)
library(tidyverse)


# File for 2016
file_2016 <- "4.outliers_3.d.ii2016_poolfstat_SNP_FST_with_locations_95.csv"
FST_all_2016 <- fread(file_2016, header = TRUE, data.table = FALSE)
file_2017 <- "4.outliers_3.d.ii2017_poolfstat_SNP_FST_with_locations_95.csv"
FST_all_2017 <- fread(file_2017, header = TRUE, data.table = FALSE)
print("check columns match")
identicalcolumns<- identical(colnames(FST_all_2017), colnames(FST_all_2016)) #Checking if colnames and order are identical (and they are!)
identicalcolumns #should say "TRUE" (if name and order are identical)
FSToutliers_95_20162017<-rbind(FST_all_2016,FST_all_2017)
colnames(FSToutliers_95_20162017)
colnames(FSToutliers_95_20162017)[1] <- "population" #renaming the first column

write.table(FSToutliers_95_20162017, 
            "4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv", 
            sep = "\t",      # Specify tab as the separator
            col.names = TRUE, # Include column names
            row.names = FALSE, # Exclude row names
            quote = FALSE)   # Don't quote character strings
```
 Split Dataset-------------------------------------------------------------
``` R
library(data.table)
input_file <- "4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv"
data <- fread(input_file, sep = "\t", header = TRUE, stringsAsFactors = FALSE)

# Split the rows into two groups based on the first column ending with "2016" or "2017"
data_2016 <- data[grep("2016$", population)]
data_2017 <- data[grep("2017$", population)]

# Write the filtered data to separate files using fwrite for faster writing
fwrite(data_2016, "4.outliers_poolfstat_SNP_FST_with_locations_95_2016.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
fwrite(data_2017, "4.outliers_poolfstat_SNP_FST_with_locations_95_2017.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
```
 All SNPs ------------------------------------------------------------
*Goal:* Generate a file that lists all SNPs present in both years (whether neutral or outliers):
``` bash
awk -F'\t' 'NR==1 {print "Chromosome_Position"; for (i=2; i<=NF; i++) print $i}' 4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv > all_snps.tsv
```
 Outliers-oonly Dataset-------------------------------------------------------------
*Goal:* Separate the SNPs that are outliers in 2016 and 2017 (so generate a dataset of outlier SNPs only (no neutral) for each year)
``` bash
# Detect the 2016 outliers:
awk -F'\t' ' #the input file is tsv
NR == 1 {
    # Store column headers to print later
    for (i = 2; i <= NF; i++) header[i] = $i
    next #ignore first column (population name)
}
{
    # Loop through all columns (starting from the second column)
    for (i = 2; i <= NF; i++) {
        if ($i == "TRUE") { #if it contains "TRUE", I want it to be printed (i.e. it is an outlier)
            # Mark the column index if it contains "TRUE"
            contains_true[i] = 1
        }
    }
}
END {
    # Print the column names for those columns that have at least one "TRUE"
    print "Chromosome_Position"
    for (i = 2; i <= NF; i++) {
        if (contains_true[i]) {
            print header[i]
        }
    }
}
' 4.outliers_poolfstat_SNP_FST_with_locations_95_2016.tsv > 2016_outliers.tsv 


# Detect the 2017 outliers:
awk -F'\t' ' #the input file is tsv
NR == 1 {
    # Store column headers to print later
    for (i = 2; i <= NF; i++) header[i] = $i
    next #ignore first column (population name)
}
{
    # Loop through all columns (starting from the second column)
    for (i = 2; i <= NF; i++) {
        if ($i == "TRUE") { #if it contains "TRUE", I want it to be printed (i.e. it is an outlier)
            # Mark the column index if it contains "TRUE"
            contains_true[i] = 1
        }
    }
}
END {
    # Print the column names for those columns that have at least one "TRUE"
    print "Chromosome_Position"
    for (i = 2; i <= NF; i++) {
        if (contains_true[i]) {
            print header[i]
        }
    }
}
' 4.outliers_poolfstat_SNP_FST_with_locations_95_2017.tsv > 2017_outliers.tsv 
```
 Outliers in either 2016 or 2017 Dataset-------------------------------------------------------------
 *Goal:* Generate a file with SNPs that are outliers in 2016 and/or 2017.
 This was taken from original upset script (but not used in upset)

 ``` bash
awk -F'\t' ' # the input file is tsv
NR == 1 {
    # Store column headers to print later
    for (i = 1; i <= NF; i++) header[i] = $i
    next # ignore first column (population name)
}
{
    # Store the first column data
    column_data[NR, 1] = $1

    # Loop through all columns (starting from the second column)
    for (i = 2; i <= NF; i++) {
        if ($i == "TRUE") { # if it contains "TRUE", I want it to be printed (i.e. it is an outlier)
            # Mark the column index if it contains "TRUE"
            contains_true[i] = 1
        }
        # Store the column data
        column_data[NR, i] = $i
    }
}
END {
    # Print the header row
    printf "%s", header[1]
    for (i = 2; i <= NF; i++) {
        if (contains_true[i]) {
            printf "\t%s", header[i]
        }
    }
    print ""

    # Print the data rows for columns that have at least one "TRUE"
    for (j = 1; j <= NR; j++) {
        printf "%s", column_data[j, 1]
        for (i = 2; i <= NF; i++) {
            if (contains_true[i]) {
                printf "\t%s", column_data[j, i]
            }
	}
	print ""
    }
}
' 4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv > 4.outliers_poolfstat_SNP_FST_95_2016_2017_outliers_only_AGAIN.tsv
